In [1]:
import torch
import torch.nn as nn
torch.manual_seed(1)
rnn_layer = nn.RNN(input_size=5, hidden_size=2, num_layers=1, batch_first=True)
w_xh = rnn_layer.weight_ih_l0
w_hh = rnn_layer.weight_hh_l0
b_xh = rnn_layer.bias_ih_l0
b_hh = rnn_layer.bias_hh_l0
print('W_xh shape:', w_xh.shape)
print('W_hh shape:', w_hh.shape)
print('b_xh shape:', b_xh.shape)
print('b_hh shape:', b_hh.shape)

W_xh shape: torch.Size([2, 5])
W_hh shape: torch.Size([2, 2])
b_xh shape: torch.Size([2])
b_hh shape: torch.Size([2])


In [2]:
x_seq = torch.tensor([[1.0]*5, [2.0]*5, [3.0]*5]).float()
## output of the simple RNN:
output, hn = rnn_layer(torch.reshape(x_seq, (1, 3, 5)))
## manually computing the output:
out_man = []
for t in range(3):
    xt = torch.reshape(x_seq[t], (1, 5))
    print(f'Time step {t} =>')
    print('Input:', xt.numpy())
    ht = torch.matmul(xt, torch.transpose(w_xh, 0, 1)) + b_hh
    print('Hidden:', ht.detach().numpy())
    if t > 0:
        prev_h = out_man[t-1]
    else:
        prev_h = torch.zeros((ht.shape))
    ot = ht + torch.matmul(prev_h, torch.transpose(w_hh, 0, 1)) + b_hh
    ot = torch.tanh(ot)
    out_man.append(ot)
    print('Output (manual):', ot.detach().numpy())
    print('RNN output:', output[:, t].detach().numpy())
    print()

Time step 0 =>
Input: [[1. 1. 1. 1. 1.]]
Hidden: [[-0.3161478   0.64722455]]
Output (manual): [[-0.21046415  0.56788784]]
RNN output: [[-0.3519801   0.52525216]]

Time step 1 =>
Input: [[2. 2. 2. 2. 2.]]
Hidden: [[-0.73478645  1.2972739 ]]
Output (manual): [[-0.5741978  0.7945334]]
RNN output: [[-0.68424344  0.76074266]]

Time step 2 =>
Input: [[3. 3. 3. 3. 3.]]
Hidden: [[-1.153425   1.9473232]]
Output (manual): [[-0.8130059   0.91817397]]
RNN output: [[-0.8649416   0.90466356]]



In [3]:
from torchtext.datasets import IMDB
train_dataset = IMDB(split='train')
test_dataset = IMDB(split='test')

In [31]:
test_dataset = list(test_dataset)

In [4]:
## Step 1: create the datasets
from torch.utils.data.dataset import random_split
torch.manual_seed(1)
train_dataset, valid_dataset = random_split(
    list(train_dataset), [20000, 5000]
)

In [5]:
## Step 2: find unique tokens (words)
import re
from collections import Counter, OrderedDict

def tokenizer(text):
    text = re.sub('<[^>]*>', '', text)
    emoticons = re.findall(
        '(?::|;|=)(?:-)?(?:\)|\(|D|P)', text.lower()
    )
    text = re.sub('[\W]+', ' ', text.lower()) + ' '.join(emoticons).replace('-', '')
    tokenized = text.split()
    return tokenized

token_counts = Counter()
for label, line in train_dataset:
    tokens = tokenizer(line)
    token_counts.update(tokens)
print('Vocab-size:', len(token_counts))

Vocab-size: 69023


In [6]:
## Step 3: encoding each unique token into integers
from sympy import ordered
from torchtext.vocab import vocab

sorted_by_freq_tuples = sorted(
    token_counts.items(), key=lambda x: x[1], reverse=True
)
ordered_dict = OrderedDict(sorted_by_freq_tuples)
vocab = vocab(ordered_dict)
vocab.insert_token('<pad>', 0)
vocab.insert_token('<unk>', 1)
vocab.set_default_index(1)

In [7]:
print([vocab[token] for token in ['this', 'is', 'an', 'example']])

[11, 7, 35, 457]


In [8]:
## Step 3-A: define the functions for transformation
text_pipeline = lambda x: [vocab[token] for token in tokenizer(x)]
label_pipeline = lambda x: 1. if x == 'pos' else 0.

In [9]:
## Step 3-B: wrap the encode and transformation function
def collate_batch(batch):
    label_list, text_list, lengths = [], [], []
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        processed_text = torch.tensor(text_pipeline(_text), dtype=torch.int64)
        text_list.append(processed_text)
        lengths.append(processed_text.size(0))
    label_list = torch.tensor(label_list)
    lengths = torch.tensor(lengths)
    padded_text_list = nn.utils.rnn.pad_sequence(
        text_list, batch_first=True
    )
    return padded_text_list, label_list, lengths

## Take a small batch
from torch.utils.data import DataLoader
dataloader = DataLoader(train_dataset, batch_size=4, shuffle=False, collate_fn = collate_batch)

In [10]:
text_batch, label_batch, length_batch = next(iter(dataloader))
print(text_batch)

tensor([[   35,  1739,     7,   449,   721,     6,   301,     4,   787,     9,
             4,    18,    44,     2,  1705,  2460,   186,    25,     7,    24,
           100,  1874,  1739,    25,     7, 34415,  3568,  1103,  7517,   787,
             5,     2,  4991, 12401,    36,     7,   148,   111,   939,     6,
         11598,     2,   172,   135,    62,    25,  3199,  1602,     3,   928,
          1500,     9,     6,  4601,     2,   155,    36,    14,   274,     4,
         42945,     9,  4991,     3,    14, 10296,    34,  3568,     8,    51,
           148,    30,     2,    58,    16,    11,  1893,   125,     6,   420,
          1214,    27, 14542,   940,    11,     7,    29,   951,    18,    17,
         15994,   459,    34,  2480, 15211,  3713,     2,   840,  3200,     9,
          3568,    13,   107,     9,   175,    94,    25,    51, 10297,  1796,
            27,   712,    16,     2,   220,    17,     4,    54,   722,   238,
           395,     2,   787,    32,    27,  5236,  

In [11]:
print(label_batch)

tensor([0., 0., 0., 0.])


In [12]:
print(length_batch)

tensor([165,  86, 218, 145])


In [13]:
print(text_batch.shape)

torch.Size([4, 218])


In [32]:
batch_size = 32
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_batch)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)
test_dl = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_batch)

In [15]:
embedding = nn.Embedding(
    num_embeddings=10, embedding_dim=3, padding_idx=0
)
# a batch of 2 samples of 4 indices each
text_encoded_input = torch.LongTensor([[1,2,4,5], [4,3,2,0]])
print(embedding(text_encoded_input))

tensor([[[ 0.7039, -0.8321, -0.4651],
         [-0.3203,  2.2408,  0.5566],
         [-0.4643,  0.3046,  0.7046],
         [-0.7106, -0.2959,  0.8356]],

        [[-0.4643,  0.3046,  0.7046],
         [ 0.0946, -0.3531,  0.9124],
         [-0.3203,  2.2408,  0.5566],
         [ 0.0000,  0.0000,  0.0000]]], grad_fn=<EmbeddingBackward0>)


In [16]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=2, batch_first=True)
        # self.rnn = nn.GRU(input_size, hidden_size, num_layers, batch_first=True)
        # self.rnn = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    
    def forward(self, x):
        _, hidden = self.rnn(x)
        out = hidden[-1, :, :] # we use the final hidden state from the last hidden layer as the input to the fc
        out = self.fc(out)
        return out
    
model = RNN(64, 32)
print(model)
model(torch.randn(5, 3, 64))

RNN(
  (rnn): RNN(64, 32, num_layers=2, batch_first=True)
  (fc): Linear(in_features=32, out_features=1, bias=True)
)


tensor([[ 0.3183],
        [ 0.1230],
        [ 0.1772],
        [-0.1052],
        [-0.1259]], grad_fn=<AddmmBackward0>)

In [17]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc1 = nn.Linear(rnn_hidden_size, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        out, (hidden, cell) = self.rnn(out)
        out = hidden[-1, :, :]
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

In [18]:
vocab_size = len(vocab)
embed_dim = 20
rnn_hidden_size = 64
fc_hidden_size = 64
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True)
  (fc1): Linear(in_features=64, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [27]:
import torch

# Pick GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Make sure model is on device
model = model.to(device)

def train(dataloader):
    model.train()
    total_acc, total_loss = 0, 0
    for i, (text_batch, label_batch, lengths) in enumerate(dataloader):
        # Move data to GPU
        text_batch = text_batch.to(device)
        label_batch = label_batch.to(device)
        lengths = lengths.to(device)

        optimizer.zero_grad()
        pred = model(text_batch, lengths)[:, 0]
        loss = loss_fn(pred, label_batch)
        loss.backward()
        optimizer.step()

        total_acc += ((pred >= 0.5).float() == label_batch).float().sum().item()
        total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)


def evaluate(dataloader):
    model.eval()
    total_acc, total_loss = 0, 0
    with torch.no_grad():
        for text_batch, label_batch, lengths in dataloader:
            # Move data to GPU
            text_batch = text_batch.to(device)
            label_batch = label_batch.to(device)
            lengths = lengths.to(device)

            pred = model(text_batch, lengths)[:, 0]
            loss = loss_fn(pred, label_batch)

            total_acc += ((pred >= 0.5).float() == label_batch).float().sum().item()
            total_loss += loss.item() * label_batch.size(0)

    return total_acc / len(dataloader.dataset), total_loss / len(dataloader.dataset)


Using device: cuda


In [28]:
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [29]:
num_epochs = 10
torch.manual_seed(1)
for epoch in range(num_epochs):
    acc_train, loss_train = train(train_dl)
    acc_valid, loss_valid = evaluate(valid_dl)
    print(f'Epoch {epoch} accuracy: {acc_train:.4f} val_accuracy: {acc_valid:.4f}')

Epoch 0 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 1 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 2 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 3 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 4 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 5 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 6 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 7 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 8 accuracy: 1.0000 val_accuracy: 1.0000
Epoch 9 accuracy: 1.0000 val_accuracy: 1.0000


In [33]:
acc_test, _ = evaluate(test_dl)
print(f'test_accuracy: {acc_test:.4f}')

test_accuracy: 1.0000


In [34]:
class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, embed_dim, padding_idx=0
        )
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(rnn_hidden_size*2, fc_hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(fc_hidden_size, 1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, text, lengths):
        out = self.embedding(text)
        out = nn.utils.rnn.pack_padded_sequence(
            out, lengths.cpu().numpy(), enforce_sorted=False, batch_first=True
        )
        _, (hidden, cell) = self.rnn(out)
        out = torch.cat((hidden[-2, :, :], hidden[-1, :, :]), dim=1)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.sigmoid(out)
        return out

In [35]:
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size, fc_hidden_size)
model

RNN(
  (embedding): Embedding(69025, 20, padding_idx=0)
  (rnn): LSTM(20, 64, batch_first=True, bidirectional=True)
  (fc1): Linear(in_features=128, out_features=64, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=64, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [36]:
import numpy as np

## Reading and processing text
with open('1268-0.txt', 'r', encoding='utf8') as fp:
    text = fp.read()
start_indx = text.find('THE MYSTERIOUS ISLAND')
end_indx = text.find('End of the Project Gutenberg')
text = text[start_indx:end_indx]
char_set = set(text)
print(f'Total Length: {len(text)}')
print('Unique Characters:', len(char_set))

Total Length: 1130779
Unique Characters: 86


In [37]:
chars_sorted = sorted(char_set)
char2int = {ch:i for i, ch in enumerate(chars_sorted)}
char_array = np.array(chars_sorted)
text_encoded = np.array(
    [char2int[ch] for ch in text], dtype=np.int32
)
print('Text encoded shape:', text_encoded.shape)

Text encoded shape: (1130779,)


In [38]:
print(text[:15], '== Encoding ==>', text_encoded[:15])
print(text_encoded[15:21], '== Reverse ==>', ''.join(char_array[text_encoded[15:21]]))

THE MYSTERIOUS  == Encoding ==> [46 34 31  1 39 51 45 46 31 44 35 41 47 45  1]
[35 45 38 27 40 30] == Reverse ==> ISLAND


In [40]:
for ex in text_encoded[:5]:
    print('{} -> {}'.format(ex, char_array[ex]))

46 -> T
34 -> H
31 -> E
1 ->  
39 -> M


In [42]:
import torch
from torch.utils.data import Dataset

seq_length = 40
chunk_size = seq_length + 1
text_chunks = [text_encoded[i:i+chunk_size] for i in range(len(text_encoded)-chunk_size)]

In [44]:
from torch.utils.data import Dataset

class TextDataset(Dataset):
    def __init__(self, text_chunks):
        self.text_chunks = text_chunks
    
    def __len__(self):
        return len(self.text_chunks)
    
    def __getitem__(self, idx):
        text_chunk = self.text_chunks[idx]
        return text_chunk[:-1].long(), text_chunk[1:].long()


seq_dataset = TextDataset(torch.tensor(text_chunks))

In [46]:
for i, (seq, target) in enumerate(seq_dataset):
    print('Input (x): ', repr(''.join(char_array[seq])))
    print('Target (y): ', repr(''.join(char_array[target])))
    print()
    if i == 1:
        break

Input (x):  'THE MYSTERIOUS ISLAND ***\n\nTHE MYSTERIOU'
Target (y):  'HE MYSTERIOUS ISLAND ***\n\nTHE MYSTERIOUS'

Input (x):  'HE MYSTERIOUS ISLAND ***\n\nTHE MYSTERIOUS'
Target (y):  'E MYSTERIOUS ISLAND ***\n\nTHE MYSTERIOUS '



In [60]:
from torch.utils.data import DataLoader

batch_size = 64
torch.manual_seed(1)
seq_dl = DataLoader(seq_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

In [67]:
import torch
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class RNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, rnn_hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn_hidden_size = rnn_hidden_size
        self.rnn = nn.LSTM(embed_dim, rnn_hidden_size, batch_first=True)
        self.fc = nn.Linear(rnn_hidden_size, vocab_size)

    def forward(self, x, hidden, cell):
        out = self.embedding(x).unsqueeze(1)
        out, (hidden, cell) = self.rnn(out, (hidden, cell))
        out = self.fc(out).reshape(out.size(0), -1)
        return out, hidden, cell

    def init_hidden(self, batch_size, device):
        hidden = torch.zeros(1, batch_size, self.rnn_hidden_size, device=device)
        cell = torch.zeros(1, batch_size, self.rnn_hidden_size, device=device)
        return hidden, cell

In [71]:
vocab_size = len(char_array)
embed_dim = 256
rnn_hidden_size = 512
torch.manual_seed(1)
model = RNN(vocab_size, embed_dim, rnn_hidden_size).to(device)
model

RNN(
  (embedding): Embedding(86, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=86, bias=True)
)

In [72]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [75]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 10000
torch.manual_seed(1)

for epoch in range(num_epochs):
    # Get a batch from the dataloader
    seq_batch, target_batch = next(iter(seq_dl))

    # Move data to device
    seq_batch = seq_batch.to(device)
    target_batch = target_batch.to(device)

    # Initialize hidden & cell states on the same device
    hidden, cell = model.init_hidden(seq_batch.size(0), device)

    optimizer.zero_grad()
    loss = 0

    for c in range(seq_length):
        pred, hidden, cell = model(seq_batch[:, c], hidden, cell)
        loss += loss_fn(pred, target_batch[:, c])

    loss.backward()
    optimizer.step()

    loss_value = loss.item() / seq_length
    if epoch % 500 == 0:
        print(f'Epoch {epoch} Loss: {loss_value:.4f}')


Epoch 0 Loss: 1.6313
Epoch 500 Loss: 1.4038
Epoch 1000 Loss: 1.3249
Epoch 1500 Loss: 1.2749
Epoch 2000 Loss: 1.2519
Epoch 2500 Loss: 1.1661
Epoch 3000 Loss: 1.1840
Epoch 3500 Loss: 1.1652
Epoch 4000 Loss: 1.1716
Epoch 4500 Loss: 1.1118
Epoch 5000 Loss: 1.1448
Epoch 5500 Loss: 1.0652
Epoch 6000 Loss: 1.1203
Epoch 6500 Loss: 1.0843
Epoch 7000 Loss: 1.0338
Epoch 7500 Loss: 1.0590
Epoch 8000 Loss: 1.0384
Epoch 8500 Loss: 1.0214
Epoch 9000 Loss: 1.0133
Epoch 9500 Loss: 1.0244


In [76]:
from torch.distributions.categorical import Categorical
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])
print('Probabilities:', nn.functional.softmax(logits, dim=1).numpy()[0])

Probabilities: [0.33333334 0.33333334 0.33333334]


In [77]:
m = Categorical(logits=logits)
samples = m.sample((10,))
print(samples.numpy())

[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


In [78]:
torch.manual_seed(1)
logits = torch.tensor([[1.0, 1.0, 1.0]])
print('Probabilities:', nn.functional.softmax(logits, dim=1).numpy()[0])


Probabilities: [0.33333334 0.33333334 0.33333334]


In [79]:
m = Categorical(logits=logits)
samples = m.sample((10,))
print(samples.numpy())

[[0]
 [0]
 [0]
 [0]
 [1]
 [0]
 [1]
 [2]
 [1]
 [1]]


In [84]:
def sample(model, starting_str, len_generated_text=500, scale_factor=1.0, device="cuda"):
    encoded_input = torch.tensor([char2int[s] for s in starting_str], device=device)
    encoded_input = torch.reshape(encoded_input, (1, -1))  # already on device
    generated_str = starting_str

    model.eval()
    hidden, cell = model.init_hidden(1, device=device)

    # warm up the RNN with the starting string
    for c in range(len(starting_str) - 1):
        _, hidden, cell = model(encoded_input[:, c].view(1), hidden, cell)

    last_char = encoded_input[:, -1]
    for i in range(len_generated_text):
        logits, hidden, cell = model(last_char.view(1), hidden, cell)
        logits = torch.squeeze(logits, 0)

        scaled_logits = logits * scale_factor
        m = Categorical(logits=scaled_logits)

        last_char = m.sample()
        generated_str += str(char_array[last_char.item()])  # .item() for indexing

    return generated_str


In [85]:
torch.manual_seed(1)
print(sample(model, starting_str='The island'))

The island was crowned it with a wall. But
the cart, the coal was contented thus.

The engineer according to the shore. Then, divided explodishing.

The engineer traped over what he could not catch behind the “Nautilus and a
foot.

Herbert emergivery member, Gideon Spilett would heard them they had no great-micals. As to the engineer, “this is in fierce of the day should not be a serious Project Gutenberg” crief or two inclination, he were able to meet them particularly invented this open yet greet liquid


In [86]:
logits = torch.tensor([[1.0, 1.0, 3.0]])
print('Probabilities before scaling:', nn.functional.softmax(logits, dim=1).numpy()[0])

Probabilities before scaling: [0.10650698 0.10650698 0.78698605]


In [87]:
print('Probabilities after scaling with 0.5:', nn.functional.softmax(logits*0.5, dim=1).numpy()[0])

Probabilities after scaling with 0.5: [0.21194156 0.21194156 0.57611686]


In [88]:
print('Probabilities after scaling with 0.1:', nn.functional.softmax(0.1*logits, dim=1).numpy()[0])

Probabilities after scaling with 0.1: [0.3104238  0.3104238  0.37915248]


In [89]:
torch.manual_seed(1)
print(sample(model, starting_str='The island', scale_factor=2.0))

The island was covered with the horizon, then to flow the cart will be entirely constituted the island. The settlers had not been seen that the convicts had taken the corral. The engineer drew them by the temperature every one of the sun was to be recognized the colonists were all the trees which could not be made to the scene of the catastrophe, for the convicts were here all the trees not to find him, and the settlers had returned to the notice of the volcano. But the cart was made of the first time the


RNN(
  (embedding): Embedding(86, 256)
  (rnn): LSTM(256, 512, batch_first=True)
  (fc): Linear(in_features=512, out_features=86, bias=True)
)